# Day 049 Project: Evaluate & Improve the Model

## What You're Building

A full evaluation report that compares a regression model and a classification model, detects overfitting, and saves a two-panel chart.

## Project Requirements

1. Evaluate a `LinearRegression` on housing data using `ModelEvaluator(task='regression')`
2. Evaluate a `LogisticRegression` on exam data using `ModelEvaluator(task='classification')`
3. Build an `overfitting_report` for the housing data (depths 1–15)
4. Store: `reg_summary` (str), `clf_summary` (str), `overfit_df` (DataFrame)
5. Save a 2-panel matplotlib figure to `model_evaluation.png`:
   - Left panel: overfitting curve (train_r2 and test_r2 vs depth)
   - Right panel: confusion matrix heatmap for the classifier
6. Run `_run_project_checks()` to verify

Use `StandardScaler` on both datasets before training.

## Provided: All Implementations

In [ ]:
import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import (r2_score, mean_squared_error, mean_absolute_error,
                              accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings('ignore')


def make_regression_data(n: int = 200, seed: int = 42) -> pd.DataFrame:
    """Numeric-only housing dataset (area, bedrooms, age → price)."""
    rng = np.random.default_rng(seed)
    area     = rng.uniform(500, 3000, n).round(0)
    bedrooms = rng.integers(1, 6, n)
    age      = rng.uniform(0, 50, n).round(1)
    price    = (area * 150 + bedrooms * 10_000 - age * 1_000
                + rng.standard_normal(n) * 10_000).round(-2)
    return pd.DataFrame({'area': area.astype(int), 'bedrooms': bedrooms,
                         'age': age, 'price': price.astype(int)})


def make_classification_data(n: int = 200, seed: int = 42) -> pd.DataFrame:
    """Student exam dataset: hours_studied + hours_sleep → passed (0/1)."""
    rng          = np.random.default_rng(seed)
    hours_studied = rng.uniform(0, 10, n).round(1)
    hours_sleep   = rng.uniform(4, 10, n).round(1)
    noise         = rng.standard_normal(n)
    score         = 1.5 * hours_studied + 0.5 * hours_sleep + noise
    passed        = (score > 9.0).astype(int)
    return pd.DataFrame({'hours_studied': hours_studied,
                         'hours_sleep':   hours_sleep,
                         'passed':        passed})


def cross_validate_model(model, X: pd.DataFrame, y: pd.Series,
                          cv: int = 5,
                          scoring: str = 'r2') -> dict:
    """K-fold cross-validation returning per-fold scores and summary stats."""
    kf     = KFold(n_splits=cv, shuffle=True, random_state=42)
    scores = cross_val_score(model, X, y, cv=kf, scoring=scoring)
    return {
        'scores':   scores,
        'mean':     round(float(scores.mean()), 4),
        'std':      round(float(scores.std()),  4),
        'min':      round(float(scores.min()),  4),
        'max':      round(float(scores.max()),  4),
        'cv_folds': cv,
        'scoring':  scoring,
    }


def overfitting_report(X: pd.DataFrame, y: pd.Series,
                        max_depths=range(1, 11),
                        test_size: float = 0.2,
                        random_state: int = 42) -> pd.DataFrame:
    """Train DecisionTreeRegressors at each depth; return train vs test R² table."""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )
    records = []
    for depth in max_depths:
        m = DecisionTreeRegressor(max_depth=depth, random_state=42)
        m.fit(X_train, y_train)
        tr_r2 = float(r2_score(y_train, m.predict(X_train)))
        te_r2 = float(r2_score(y_test,  m.predict(X_test)))
        gap   = round(tr_r2 - te_r2, 4)
        records.append({
            'max_depth': depth,
            'train_r2':  round(tr_r2, 4),
            'test_r2':   round(te_r2, 4),
            'gap':       gap,
            'overfit':   bool(gap > 0.1),
        })
    return pd.DataFrame(records)


def train_classifier(X_train: pd.DataFrame,
                     y_train: pd.Series,
                     max_iter: int = 1000) -> LogisticRegression:
    """Fit LogisticRegression and return the fitted model."""
    model = LogisticRegression(random_state=42, max_iter=max_iter)
    model.fit(X_train, y_train)
    return model


def classification_metrics(model, X_test: pd.DataFrame,
                            y_test: pd.Series) -> dict:
    """Full classification evaluation: accuracy, precision, recall, F1, matrix, report."""
    y_pred = model.predict(X_test)
    return {
        'accuracy':         round(float(accuracy_score(y_test, y_pred)), 4),
        'precision':        round(float(precision_score(y_test, y_pred,
                                                         zero_division=0)), 4),
        'recall':           round(float(recall_score(y_test, y_pred,
                                                      zero_division=0)), 4),
        'f1':               round(float(f1_score(y_test, y_pred,
                                                  zero_division=0)), 4),
        'confusion_matrix': confusion_matrix(y_test, y_pred),
        'report':           classification_report(y_test, y_pred),
    }


class ModelEvaluator:
    """
    Unified evaluator for regression and classification models.

    Usage (regression):
        ev     = ModelEvaluator(task='regression')
        result = ev.evaluate(LinearRegression(), X_train, X_test, y_train, y_test)
        print(ev.summary())

    Usage (classification):
        ev     = ModelEvaluator(task='classification')
        result = ev.evaluate(LogisticRegression(), X_train, X_test, y_train, y_test)
    """

    def __init__(self, task: str = 'regression'):
        assert task in ('regression', 'classification'), \
            f"task must be 'regression' or 'classification', got {task!r}"
        self.task     = task
        self._results = {}

    def evaluate(self, model, X_train: pd.DataFrame, X_test: pd.DataFrame,
                 y_train: pd.Series, y_test: pd.Series, cv: int = 5) -> dict:
        """Cross-validate on train, fit, then evaluate on test."""
        scoring   = 'r2' if self.task == 'regression' else 'accuracy'
        cv_result = cross_validate_model(model, X_train, y_train,
                                         cv=cv, scoring=scoring)
        model.fit(X_train, y_train)
        if self.task == 'regression':
            y_pred  = model.predict(X_test)
            r2      = float(r2_score(y_test, y_pred))
            rmse    = float(np.sqrt(mean_squared_error(y_test, y_pred)))
            mae     = float(mean_absolute_error(y_test, y_pred))
            metrics = {'r2': round(r2, 4),
                       'rmse': round(rmse, 2),
                       'mae':  round(mae,  2)}
        else:
            metrics = classification_metrics(model, X_test, y_test)
        self._results = {'task': self.task, 'cv': cv_result, 'metrics': metrics}
        return self._results

    def summary(self) -> str:
        """Return a formatted multi-line summary string."""
        if not self._results:
            return 'No evaluation run yet.'
        cv = self._results['cv']
        m  = self._results['metrics']
        lines = [
            f"Task: {self.task}",
            f"Cross-val {cv['scoring']} ({cv['cv_folds']}-fold): "
            f"{cv['mean']:.4f} \u00b1 {cv['std']:.4f}",
        ]
        if self.task == 'regression':
            lines.append(f"Test  R\u00b2={m['r2']:.4f}  "
                         f"RMSE={m['rmse']:.2f}  MAE={m['mae']:.2f}")
        else:
            lines.append(f"Test  Acc={m['accuracy']:.4f}  F1={m['f1']:.4f}")
        return '\n'.join(lines)

## Your Pipeline

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ── Regression evaluation ──
df_r = make_regression_data(200)
X_r  = df_r.drop(columns=['price'])
y_r  = df_r['price']
X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(X_r, y_r,
                                                    test_size=0.2, random_state=42)
sc_r    = StandardScaler()
X_tr_rs = pd.DataFrame(sc_r.fit_transform(X_tr_r), columns=X_r.columns)
X_te_rs = pd.DataFrame(sc_r.transform(X_te_r),     columns=X_r.columns)

# TODO: ev_r        = ModelEvaluator(task='regression')
# TODO: ev_r.evaluate(LinearRegression(), X_tr_rs, X_te_rs, y_tr_r, y_te_r)
# TODO: reg_summary = ev_r.summary()
# TODO: print(reg_summary)

# ── Classification evaluation ──
df_c = make_classification_data(200)
X_c  = df_c.drop(columns=['passed'])
y_c  = df_c['passed']
X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(X_c, y_c,
                                                    test_size=0.2, random_state=42)
sc_c    = StandardScaler()
X_tr_cs = pd.DataFrame(sc_c.fit_transform(X_tr_c), columns=X_c.columns)
X_te_cs = pd.DataFrame(sc_c.transform(X_te_c),     columns=X_c.columns)

# TODO: ev_c        = ModelEvaluator(task='classification')
# TODO: ev_c.evaluate(LogisticRegression(max_iter=1000), X_tr_cs, X_te_cs, y_tr_c, y_te_c)
# TODO: clf_summary = ev_c.summary()
# TODO: print(clf_summary)

# ── Overfitting report ──
# TODO: overfit_df = overfitting_report(X_r, y_r, max_depths=range(1, 16))
# TODO: print(overfit_df.to_string(index=False))

# ── Save 2-panel figure ──
# TODO: fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
# TODO: # Left: overfitting curve
# TODO: ax1.plot(overfit_df['max_depth'], overfit_df['train_r2'],
#                label='Train R\u00b2', marker='o')
# TODO: ax1.plot(overfit_df['max_depth'], overfit_df['test_r2'],
#                label='Test R\u00b2', marker='s')
# TODO: ax1.set_xlabel('max_depth'); ax1.set_ylabel('R\u00b2')
# TODO: ax1.set_title('Overfitting Curve'); ax1.legend()
# TODO: # Right: confusion matrix heatmap
# TODO: cm = ev_c._results['metrics']['confusion_matrix']
# TODO: ax2.imshow(cm, cmap='Blues')
# TODO: for i in range(2):
#     for j in range(2):
#         ax2.text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=14)
# TODO: ax2.set_xticks([0,1]); ax2.set_yticks([0,1])
# TODO: ax2.set_xticklabels(['Pred 0','Pred 1'])
# TODO: ax2.set_yticklabels(['True 0','True 1'])
# TODO: ax2.set_title('Confusion Matrix')
# TODO: plt.tight_layout()
# TODO: fig.savefig('model_evaluation.png', bbox_inches='tight', dpi=100)
# TODO: plt.close('all')
# TODO: print('Chart saved: model_evaluation.png')

## Checks

In [ ]:
import os

def _run_project_checks():
    total = 5
    passed = 0

    # Check 1: reg_summary defined and is a string
    try:
        assert 'reg_summary' in globals(), \
            'reg_summary not defined — call ev_r.summary()'
        assert isinstance(reg_summary, str) and len(reg_summary) > 10
        passed += 1; print('\u2705 Check 1: reg_summary defined')
        print(reg_summary)
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: clf_summary defined and is a string
    try:
        assert 'clf_summary' in globals(), \
            'clf_summary not defined — call ev_c.summary()'
        assert isinstance(clf_summary, str) and len(clf_summary) > 10
        passed += 1; print('\u2705 Check 2: clf_summary defined')
        print(clf_summary)
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: overfit_df has 15 rows (depths 1-15)
    try:
        assert 'overfit_df' in globals(), \
            'overfit_df not defined — call overfitting_report(...)'
        assert isinstance(overfit_df, pd.DataFrame)
        assert len(overfit_df) == 15, \
            f'overfit_df should have 15 rows (depths 1-15), got {len(overfit_df)}'
        passed += 1; print(f'\u2705 Check 3: overfit_df has 15 rows')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: cv mean in reg_summary suggests R² > 0
    try:
        assert 'r2' in reg_summary.lower() or 'R²' in reg_summary, \
            'reg_summary should mention R\u00b2'
        passed += 1; print('\u2705 Check 4: reg_summary mentions R\u00b2')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: chart file saved
    try:
        assert os.path.exists('model_evaluation.png'), \
            'model_evaluation.png not found'
        assert os.path.getsize('model_evaluation.png') > 1000
        passed += 1; print('\u2705 Check 5: model_evaluation.png saved')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Project complete!')
    print(f'\nScore: {passed}/{total}')


_run_project_checks()

## Bonus Challenges

- Add `DecisionTreeClassifier(max_depth=3)` to the classification pipeline and compare its accuracy and F1 against LogisticRegression
- Try `scoring='neg_root_mean_squared_error'` in cross_val_score (sklearn uses negative scores so higher is always better) and convert back to positive
- Plot the learning curve: train on 20%, 40%, 60%, 80%, 100% of training data and show how test R² changes
- Use `ollama.chat` to narrate the reg_summary and clf_summary in plain English